<a href="https://colab.research.google.com/github/25wh1a05ce-pixel/EcoHackers/blob/main/No-burn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q tensorflow kagglehub

In [3]:
import kagglehub

path = kagglehub.dataset_download(
    "mostafaabla/garbage-classification"
)

print("Dataset downloaded to:")
print(path)

100%|██████████| 239M/239M [00:03<00:00, 64.8MB/s]

Extracting files...


Dataset downloaded to:
/root/.cache/kagglehub/datasets/mostafaabla/garbage-classification/versions/1


In [4]:
import os

print("Dataset path:")
print(path)

print("\nFiles and folders:")

for item in os.listdir(path):
    print(item)

Dataset path:
/root/.cache/kagglehub/datasets/mostafaabla/garbage-classification/versions/1

Files and folders:
garbage_classification


In [5]:
import os

dataset_path = os.path.join(path, "garbage_classification")

print("Waste categories:\n")

for category in sorted(os.listdir(dataset_path)):
    category_path = os.path.join(dataset_path, category)

    if os.path.isdir(category_path):
        number_of_images = len(os.listdir(category_path))
        print(f"{category} : {number_of_images} images")

Waste categories:

battery : 945 images
biological : 985 images
brown-glass : 607 images
cardboard : 891 images
clothes : 5325 images
green-glass : 629 images
metal : 769 images
paper : 1050 images
plastic : 865 images
shoes : 1977 images
trash : 697 images
white-glass : 775 images


In [6]:
import os
import shutil

# Original dataset
source_path = dataset_path

# New dataset location
prepared_path = "/content/waste_dataset"

# Create folders
categories = [
    "Plastic",
    "Organic",
    "Paper_Cardboard",
    "Glass",
    "Metal",
    "E_Waste"
]

for category in categories:
    os.makedirs(
        os.path.join(prepared_path, category),
        exist_ok=True
    )

print("Folders created successfully!")

# Mapping original dataset → our 6 classes
mapping = {
    "plastic": "Plastic",
    "biological": "Organic",
    "paper": "Paper_Cardboard",
    "cardboard": "Paper_Cardboard",
    "brown-glass": "Glass",
    "green-glass": "Glass",
    "white-glass": "Glass",
    "metal": "Metal",
    "battery": "E_Waste"
}

# Copy images
for source_category, target_category in mapping.items():

    source_folder = os.path.join(
        source_path,
        source_category
    )

    target_folder = os.path.join(
        prepared_path,
        target_category
    )

    for filename in os.listdir(source_folder):

        source_file = os.path.join(
            source_folder,
            filename
        )

        target_file = os.path.join(
            target_folder,
            filename
        )

        shutil.copy2(
            source_file,
            target_file
        )

print("\nDataset preparation completed!")

Folders created successfully!

Dataset preparation completed!


In [7]:
import os

print("Prepared dataset:\n")

for category in sorted(os.listdir(prepared_path)):
    folder = os.path.join(prepared_path, category)

    if os.path.isdir(folder):
        count = len(os.listdir(folder))
        print(f"{category}: {count} images")

Prepared dataset:

E_Waste: 945 images
Glass: 2011 images
Metal: 769 images
Organic: 985 images
Paper_Cardboard: 1941 images
Plastic: 865 images


In [8]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    prepared_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    prepared_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("\nClasses:")
print(class_names)

print("\nTraining batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

Found 7516 files belonging to 6 classes.
Using 6013 files for training.
Found 7516 files belonging to 6 classes.
Using 1503 files for validation.

Classes:
['E_Waste', 'Glass', 'Metal', 'Organic', 'Paper_Cardboard', 'Plastic']

Training batches: 188
Validation batches: 47


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ---------------------------------------
# DATA PERFORMANCE
# ---------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(
    buffer_size=AUTOTUNE
)

val_ds = val_ds.prefetch(
    buffer_size=AUTOTUNE
)


# ---------------------------------------
# DATA AUGMENTATION
# ---------------------------------------

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


# ---------------------------------------
# PRETRAINED MODEL
# ---------------------------------------

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)


# Freeze pretrained layers

base_model.trainable = False


# ---------------------------------------
# BUILD CLASSIFIER
# ---------------------------------------

model = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    data_augmentation,

    layers.Rescaling(
        1.0 / 127.5,
        offset=-1
    ),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.2),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.2),

    layers.Dense(
        6,
        activation="softmax"
    )

])


# ---------------------------------------
# COMPILE
# ---------------------------------------

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


# ---------------------------------------
# SHOW MODEL
# ---------------------------------------

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 376s 2s/step - accuracy: 0.9107 - loss: 0.2674 - val_accuracy: 0.9348 - val_loss: 0.2095
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 363s 2s/step - accuracy: 0.9105 - loss: 0.2531 - val_accuracy: 0.9261 - val_loss: 0.2078
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - accuracy: 0.9105 - loss: 0.2509 - val_accuracy: 0.9288 - val_loss: 0.2087
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 414s 2s/step - accuracy: 0.9120 - loss: 0.2448 - val_accuracy: 0.9335 - val_loss: 0.2035
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 372s 2s/step - accuracy: 0.9135 - loss: 0.2340 - val_accuracy: 0.9328 - val_loss: 0.2020
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 348s 2s/step - accuracy: 0.9200 - loss: 0.2233 - val_accuracy: 0.9335 - val_loss: 0.1997
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 382s 2s/step - accuracy: 0.9233 - loss: 0.2142 - val_accuracy: 0.9375 - val_loss: 0.1944
Epoch 8/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 371s 2s/step - accuracy: 0.9207 - loss: 0.2161 - val_accu

In [12]:
# Evaluate the model on validation images

val_loss, val_accuracy = model.evaluate(val_ds)

print(f"\nValidation Accuracy: {val_accuracy * 100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

47/47 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - accuracy: 0.9395 - loss: 0.1922

Validation Accuracy: 93.95%
Validation Loss: 0.1922


In [13]:
model.save("/content/waste_classifier.keras")

print("Model saved successfully!")

Model saved successfully!


In [25]:
from google.colab import files
from PIL import Image
import numpy as np

uploaded = files.upload()

image_path = list(uploaded.keys())[0]

img = Image.open(image_path).convert("RGB")
img = img.resize((224, 224))

img_array = np.array(img)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array)

predicted_index = np.argmax(predictions[0])

print("\nPrediction:", class_names[predicted_index])
print(f"Confidence: {predictions[0][predicted_index] * 100:.2f}%")

print("\nAll probabilities:")
for class_name, probability in zip(class_names, predictions[0]):
    print(f"{class_name}: {probability * 100:.2f}%")

Saving e w.webp to e w.webp
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

Prediction: E_Waste
Confidence: 69.56%

All probabilities:
E_Waste: 69.56%
Glass: 0.39%
Metal: 29.62%
Organic: 0.06%
Paper_Cardboard: 0.22%
Plastic: 0.14%


In [26]:
print("Model loaded:", model is not None)
print("Classes:", class_names)

Model loaded: True
Classes: ['E_Waste', 'Glass', 'Metal', 'Organic', 'Paper_Cardboard', 'Plastic']


In [27]:
import tensorflow as tf

# Convert trained model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model
tflite_path = "/content/waste_classifier.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("TFLite model created successfully!")
print("Model size:", round(len(tflite_model) / (1024 * 1024), 2), "MB")

Saved artifact at '/tmp/tmpedk6u605'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133337103544976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103546704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103545552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103547856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103546896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103547280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103548240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103548048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103547664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133337103547472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1333371035

In [28]:
from google.colab import files

files.download("/content/waste_classifier.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>